In [242]:
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score,balanced_accuracy_score

### Load the embeddings for the C.S.sylv sulcal region

In [230]:
ukb_embeddings = pd.read_csv('/neurospin/dico/data/deep_folding/current/models/Champollion_V0_trained_on_UKB40/SC-sylv_right/11-36-10_85_0/ukb40_random_epoch80_embeddings/full_embeddings.csv', index_col=0)
#ukb_embeddings = pd.read_csv('/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/11-43-38_3/ukb40_random_epoch100_embeddings/full_embeddings.csv', index_col=0)
print(ukb_embeddings.shape)
ukb_embeddings.head()

(42433, 256)


,dim1,dim2,dim3,dim4,dim5,dim6,dim7,dim8,dim9,dim10,...,dim247,dim248,dim249,dim250,dim251,dim252,dim253,dim254,dim255,dim256
ID,,,,,,,,,,,,,,,,,,,,,
sub-1000021,144.70032,-2.33259,-55.521217,-3.811033,-44.061070,41.837160,15.210489,-3.336162,-16.129618,-20.501787,...,-12.813190,-1.744127,87.089485,15.318643,-43.634010,40.511910,21.935389,5.046747,-9.521525,-53.306515
sub-1000325,189.53354,31.76580,18.947166,-8.772030,20.106237,-33.390730,17.316332,-29.190847,20.369950,-5.060660,...,17.064726,-24.591824,33.550570,47.255363,-66.175830,47.804210,-30.778520,9.909952,-13.292032,-40.064840
sub-1000458,150.20581,-1.78613,-26.557894,8.816937,-41.133255,136.199750,-2.472998,12.573650,12.101191,22.242050,...,-11.313081,5.824224,8.352239,13.173943,23.730171,-16.312475,-6.631551,6.091625,20.440187,-85.629180
sub-1000575,145.42535,-49.58505,8.499626,12.548178,-20.224136,76.969540,4.202240,-19.305752,-32.917637,25.650387,...,-14.098648,-0.068690,-18.289234,-10.241393,18.584848,-12.241503,33.521553,1.713032,8.407868,-88.648970
sub-1000606,153.36888,29.05402,-11.328059,56.158030,-35.677720,13.826549,-7.284871,21.386320,29.033829,-8.862112,...,-15.195081,14.316205,36.025223,43.307170,1.105324,-9.904149,-24.226252,-0.112310,-18.451952,2.476232


### Reduce dimension (hope to remove the noise) with a PCA

In [231]:
n_components=11

pca = PCA(n_components=n_components)
pca.fit(ukb_embeddings)
print(pca.explained_variance_ratio_)
(np.cumsum(pca.explained_variance_ratio_) < 0.9).sum()

[0.17592706 0.12306105 0.11746931 0.1103346  0.10755888 0.09084436
 0.07942355 0.07295775 0.04263993 0.02652651 0.01563623]


8

In [232]:
ukb_pca_bdd = pca.transform(ukb_embeddings)

In [233]:
#scaler = StandardScaler()
#scaler.fit(ukb_embeddings)
#ukb_scl_bdd = scaler.transform(ukb_embeddings)
#ukb_scl_bdd

#### First approach: SVM trained to find the interruption

In [234]:
model = SVC(kernel='linear', probability=True,
            random_state=42,
            C=0.01, class_weight='balanced', decision_function_shape='ovr')

In [235]:
interrupted = [
'sub-1310920',
'sub-1376904',
'sub-2863742',
'sub-3694216',
'sub-1037052',
'sub-3250551',
'sub-5401486',
'sub-1499791', # good
'sub-1911266',
'sub-4217758',
'sub-2693192',
'sub-1633860',
'sub-5222070',
'sub-3292254',
'sub-1613821',
'sub-2771619',
'sub-3159828',
'sub-4632483',
'sub-5936108',
'sub-3794487',
'sub-1420697', # not sure
'sub-1111996', # not sure
'sub-1425827', # not sure
'sub-2846621', # good
'sub-2004479',
'sub-3891499',
'sub-5236788',
'sub-3061407', # very good
'sub-5693167',
'sub-2155264', # very good
'sub-2444973', # very good
'sub-5245412', # good
'sub-5574911', # very good
'sub-2852894', # very good
'sub-1106033', # very good
'sub-5984646', # very good
'sub-5739487', # very good
'sub-3492298', # good
'sub-5712569', # not sure
'sub-2200121', # not sure
'sub-5638090', # good
'sub-4496792', # good
'sub-5129881', # good
'sub-1775041', # good
'sub-1094593', # good
'sub-1358401', # good
'sub-4354208', # very good
'sub-3492298', # good
'sub-1428452',
'sub-5731125',
'sub-4995189', # very good
'sub-1499791',
'sub-2762943', # very good
'sub-3386408', # not sure
'sub-5665554', # not sure
'sub-1130686',
'sub-2484762', # good
]

not_interrupted = [
'sub-3264612', 
'sub-4805237', 
'sub-1422413',
'sub-3264612', 
'sub-4805237', 
'sub-1422413',
'sub-4491384', 
'sub-2946274',
'sub-5581707',
'sub-4834994',
'sub-5437419',
'sub-5054716',
'sub-2889389',
'sub-4520944',
'sub-3009279',
'sub-1190643',
'sub-5123219',
'sub-4016129',
'sub-4411765',
'sub-3234836',
'sub-5486726',
'sub-2592717',
'sub-4116944',
'sub-3670173',
'sub-1273718',
'sub-2833426',
'sub-1352284',
'sub-2389411', 
'sub-2970418', 
'sub-5605784', 
'sub-2141551', 
'sub-1979982',
'sub-5643778', 
'sub-3693543', 
'sub-4805119', 
'sub-5686761', 
'sub-2733674',
'sub-2097565',
'sub-5292898', 
'sub-2118136',
'sub-5966409',
'sub-1996092',
'sub-2036033',
'sub-3333294',
'sub-2193253',
'sub-3603191',
'sub-3936967', 
'sub-1286007', 
'sub-3013938',
'sub-5117110',
'sub-2228486',
'sub-3721299',
'sub-4420611', 
'sub-2349203',
'sub-2207793', 
'sub-2816262',
'sub-3765466'
]

In [236]:
X_train = ukb_embeddings.loc[interrupted + not_interrupted]
y_train = [1 for i in range(len(interrupted))] + [0 for i in range(len(not_interrupted))]
X_train_pca = pca.transform(X_train)
len(interrupted), len(not_interrupted)

(56, 52)

In [237]:
model.fit(X_train_pca, y_train)
roc_auc_score(y_train ,model.predict_proba(X_train_pca)[:,1]), balanced_accuracy_score(y_train, model.predict(X_train_pca))

(0.7525755494505494, 0.6593406593406593)

In [238]:
prediction = pd.DataFrame({"IID" : list(ukb_embeddings.index),
              "Pred" : model.predict_proba(ukb_pca_bdd)[:,1]})
prediction

,IID,Pred
0,sub-1000021,0.379748
1,sub-1000325,0.455877
2,sub-1000458,0.485624
3,sub-1000575,0.437963
4,sub-1000606,0.447588
...,...,...
42428,sub-6023847,0.470468
42429,sub-6024038,0.493249
42430,sub-6024150,0.382854
42431,sub-6024379,0.480923


In [239]:
prediction[prediction['IID']=='sub-2036033']

,IID,Pred
8695,sub-2036033,0.592888


In [240]:
prediction[prediction["IID"].isin(interrupted)]

,IID,Pred
267,sub-1037052,0.514677
773,sub-1094593,0.505807
863,sub-1106033,0.553839
915,sub-1111996,0.555763
1075,sub-1130686,0.643917
2589,sub-1310920,0.607122
2981,sub-1358401,0.586928
3125,sub-1376904,0.519843
3467,sub-1420697,0.576891
3505,sub-1425827,0.507361


In [241]:
((prediction[~(prediction["IID"].isin(interrupted))]).sort_values(by="Pred")[-50:-25].IID).to_list()

['sub-4387100',
 'sub-1040771',
 'sub-1107083',
 'sub-3398388',
 'sub-5965552',
 'sub-5631945',
 'sub-4359267',
 'sub-1094796',
 'sub-5390291',
 'sub-4803848',
 'sub-4602674',
 'sub-5247493',
 'sub-1006112',
 'sub-2484762',
 'sub-5307155',
 'sub-2600434',
 'sub-3650827',
 'sub-1555082',
 'sub-3008577',
 'sub-4048842',
 'sub-4315059',
 'sub-1452468',
 'sub-3313133',
 'sub-5892546',
 'sub-1664200']

#### Second approach: Euclidian distance in the reduced latent space

In [29]:
from scipy.spatial import distance

In [174]:
list_dist = [distance.euclidean(pca.transform(ukb_embeddings.loc['sub-5984646'].to_numpy().reshape(1,-1)), ukb_pca_bdd[i]) for i in range(len(ukb_pca_bdd))]
df_dist = pd.DataFrame({"IID":list(ukb_embeddings.index), "Dist":list_dist})

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr

In [178]:
(df_dist.sort_values(by='Dist').iloc[75:100].IID).to_list()

['sub-4354208',
 'sub-3474250',
 'sub-3388290',
 'sub-2144053',
 'sub-1427479',
 'sub-3413288',
 'sub-4460751',
 'sub-2324289',
 'sub-4390092',
 'sub-3722413',
 'sub-4136595',
 'sub-4884263',
 'sub-1665658',
 'sub-5282015',
 'sub-1201473',
 'sub-3932231',
 'sub-5694131',
 'sub-2004547',
 'sub-1645090',
 'sub-2747271',
 'sub-5544350',
 'sub-4310878',
 'sub-3933669',
 'sub-1674005',
 'sub-3276505']

### Visualization with Anatomist

In [ ]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims

In [ ]:
dataset = 'UkBioBank40'
region = "S.C.-sylv."
side = "R"

bucket_path = f'/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}buckets'
mm_skeleton_path = f'/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}crops'

In [ ]:
sample = ((prediction[~(prediction["IID"].isin(interrupted))]).sort_values(by="Pred")[-25:].IID).to_list()

In [ ]:
volume=True


bucket_files = []
volume_files = []

for subject_id in sample:

    bck_path = f'{bucket_path}/{subject_id}_cropped_skeleton.bck'
    volume_path = f"{mm_skeleton_path}/{subject_id}_cropped_skeleton.nii.gz"
    
    if volume:
        if os.path.isfile(volume_path):
            vol = aims.read(volume_path)
            volume_files.append(vol)
        else:
            print(f"{volume_path} is not a correct path, or the .nii.gz doesn't exist")
    else:
        if os.path. isfile(bck_path):
            bucket_files.append(bck_path)
        else:
            print(f"{bck_path} is not a correct path, or the .bck doesn't exist")

block = a.createWindowsBlock(5) # 10 columns
dic_windows = {}

if volume:
    for i, vol in enumerate(volume_files):
        dic_windows[f'a_vol{i}'] = a.toAObject(vol)
        #dic_windows[f'a_vol{i}'].setPalette(absoluteMode=True)
        dic_windows[f'rvol{i}'] = a.fusionObjects(objects=[dic_windows[f'a_vol{i}']], method='VolumeRenderingFusionMethod')
        dic_windows[f'rvol{i}'].releaseAppRef()
        dic_windows[f'wvr{i}'] = a.createWindow('3D', block=block) #geometry=[100+400*(i%3), 100+440*(i//3), 400, 400])
        dic_windows[f'wvr{i}'].addObjects(dic_windows[f'rvol{i}'])

else:
    for i, file in enumerate(bucket_files):
        dic_windows[f'bck_{i}'] = a.loadObject(file)
        dic_windows[f'w_{i}'] = a.createWindow('3D', block=block)#geometry=[100+400*(i%3), 100+440*(i//3), 400, 400])
        dic_windows[f'w_{i}'].addObjects(dic_windows[f'bck_{i}'])